# บทที่ 13 - ความจำของเอเย่นต์


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-4.1-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ประเภทของหน่วยความจำของเอเจนต์

เอเจนต์ AI สามารถใช้หน่วยความจำในรูปแบบต่าง ๆ ซึ่งแต่ละแบบมีจุดประสงค์เฉพาะตัว:

### หน่วยความจำทำงาน (Working Memory)
กระทู้การสนทนาเอง — ข้อความที่แลกเปลี่ยนกันในการสนทนาเดียว เอเจนต์สามารถอ้างอิงกลับไปยังข้อความก่อนหน้าในกระทู้เดียวกันเพื่อรักษาความสอดคล้อง ใน MAF จะสร้างหน่วยความจำนี้ผ่าน **`agent.create_session()`** ซึ่งจะคืนค่า `AgentSession`

### หน่วยความจำระยะสั้น (Short-Term Memory)
ข้อมูลที่คงอยู่ตลอดระยะเวลาของงานหรือเซสชันแต่จะไม่ถูกจัดเก็บถาวร เช่น เอเจนต์อาจรวบรวมข้อเท็จจริงระหว่างการสนทนาวางแผนหลายรอบและใช้ข้อมูลเหล่านั้นเพื่อสร้างแผนการเดินทางขั้นสุดท้าย

### หน่วยความจำระยะยาว (Long-Term Memory)
ความชอบและข้อเท็จจริงที่คงอยู่ **ข้ามเซสชัน** ผู้ใช้ที่กลับมาไม่ควรต้องบอกข้อจำกัดด้านอาหารหรือสไตล์การเดินทางซ้ำอีกครั้ง หน่วยความจำระยะยาวโดยปกติจะถูกสนับสนุนโดยที่เก็บข้อมูลภายนอก — เช่นฐานข้อมูล ไฟล์ หรือดัชนีเวกเตอร์ — และถูกนำเสนอให้เอเจนต์ผ่านเครื่องมือ


## หน่วยความจำทำงานกับเซสชัน

รูปแบบที่ง่ายที่สุดของหน่วยความจำคือเซสชันการสนทนา เมื่อคุณส่งผ่านเซสชันเดียวกัน (ที่สร้างผ่าน `agent.create_session()`) ไปยังการเรียก `agent.run()` ติดต่อกัน มันจะทำให้เอเย่นต์เห็นประวัติทั้งหมดของการสนทนานั้นและสามารถเรียกคืนรายละเอียดก่อนหน้านี้ได้

มาสร้างเอเย่นต์ท่องเที่ยวและสาธิตหน่วยความจำทำงานกันดีกว่า


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ตัวแทนจำงบประมาณได้อย่างถูกต้องเพราะข้อความทั้งสองมีเซสชันเดียวกัน นี่คือ **หน่วยความจำทำงาน** — ซึ่งมีอยู่เฉพาะตลอดช่วงเวลาของเซสชันเท่านั้น

### จะเกิดอะไรขึ้นกับกระทู้ใหม่?

ถ้าเราสร้างเซสชัน **ใหม่** ตัวแทนจะไม่มีความทรงจำในการสนทนาก่อนหน้านี้:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## รูปแบบความทรงจำระยะยาว

เพื่อจดจำความชอบของผู้ใช้ **ข้ามหลายเซสชัน** เราจำเป็นต้องมีที่เก็บข้อมูลถาวรที่อยู่นอกเหนือจากเธรดสนทนา ตัวแทนเข้าถึงที่เก็บนี้ผ่าน **เครื่องมือ** — ฟังก์ชันที่มันสามารถเรียกใช้เพื่อบันทึกและเรียกคืนข้อมูล

ด้านล่างนี้เราจะทำการสร้างที่เก็บข้อมูลความชอบในหน่วยความจำอย่างง่าย (ในงานจริงคุณจะสำรองข้อมูลนี้ด้วยฐานข้อมูลหรือดัชนีเวกเตอร์) และเปิดเผยเป็นเครื่องมือที่ตัวแทนสามารถใช้

### สถาปัตยกรรม
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### สถานการณ์ที่ 1 — ผู้ใช้ครั้งแรกจองทริปครบรอบ

ซาราห์มาเยือนครั้งแรก ตัวแทนควรบันทึกความชอบของเธอผ่านเครื่องมือและใช้เพื่อแนะนำโรงแรม


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### สถานการณ์ที่ 2 — ซาร่าห์กลับมาอีกหลายสัปดาห์ต่อมา

ซาร่าห์เริ่ม **เธรดใหม่เอี่ยม** (จำลองเซสชันใหม่) หน่วยความจำระหว่างทำงานจะว่างเปล่า แต่ร้านเก็บข้อมูลความชอบระยะยาวยังคงมีข้อมูลของเธออยู่ ตัวแทนควรดึงข้อมูลนั้นมาใช้และใช้ในการแนะนำที่ปรับให้เหมาะกับบุคคล


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้เกี่ยวกับความจำของตัวแทนสามประเภทและวิธีการใช้งานร่วมกับ Microsoft Agent Framework:

| ประเภทความจำ | กลไก MAF | ระยะเวลาใช้งาน |
|---|---|---|
| **ทำงาน** | `agent.create_session()` | การสนทนาเดียว |
| **ระยะสั้น** | บริบทที่สะสมภายในเธรด | งาน / เซสชันเดียว |
| **ระยะยาว** | ที่เก็บข้อมูลภายนอกที่เข้าถึงผ่าน `@tool` ฟังก์ชัน | ข้ามเซสชัน |

### ข้อสรุปสำคัญ
1. **`agent.create_session()`** ให้ความจำแบบทำงาน — ตัวแทนเห็นประวัติการสนทนาทั้งหมดภายในเซสชัน
2. **เซสชันใหม่สูญเสียบริบท** — หากไม่มีความจำระยะยาว ตัวแทนจะจำการสนทนาในอดีตไม่ได้
3. **ฟังก์ชัน `@tool`** เชื่อมช่องว่าง — ช่วยให้ตัวแทนบันทึกและดึงข้อมูลจากที่เก็บข้อมูลถาวร
4. **การปรับแต่งส่วนบุคคลดีขึ้นเมื่อเวลาผ่านไป** — ยิ่งเก็บความชอบมากเท่าไหร่ คำแนะนำของตัวแทนก็จะยิ่งดีขึ้นเท่านั้น

### การใช้งานในโลกจริง
- **บริการลูกค้า**: จำประวัติและความชอบของลูกค้า
- **ผู้ช่วยส่วนบุคคล**: รักษาบริบทข้ามวันหรือสัปดาห์
- **การดูแลสุขภาพ**: ติดตามข้อมูลและความชอบของผู้ป่วย
- **พาณิชย์อีเล็กทรอนิกส์**: ช้อปปิ้งแบบปรับตามบุคคลจากประวัติ

### ขั้นตอนต่อไป
- แทนที่ dict ในหน่วยความจำด้วยฐานข้อมูลหรือที่เก็บเวกเตอร์ (เช่น Azure AI Search)
- เพิ่มการหมดอายุของความจำสำหรับข้อมูลที่ต้องการความไวต่อเวลา
- สร้างระบบหลายตัวแทนที่มีหน่วยความจำร่วมกัน
- สำรวจสมุดบันทึก Cognee สำหรับความจำที่สนับสนุนด้วยกราฟความรู้


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
